# Phase 3: Vision Token Math

This notebook measures image-token behavior empirically by sending synthetic images of different sizes through the API and recording the returned usage fields.

In [1]:
required_packages = {
    "openai": "openai",
    "PIL": "pillow",
    "pandas": "pandas",
}

missing = []
for module_name, package_name in required_packages.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    raise ImportError(
        "Install these packages once in your terminal before running Phase 3: "
        + ", ".join(missing)
    )

print("Dependencies are available.")


Dependencies are available.


In [3]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image

from vision_lab_utils import (
    build_response_record,
    call_vision_json,
    create_synthetic_image,
    ensure_dir,
    get_client,
    save_json,
)

client = get_client()
ROOT = Path.cwd()
IMAGE_DIR = ensure_dir(ROOT / "images" / "token_math")
OUTPUT_DIR = ensure_dir(ROOT / "outputs")

SIZE_MATRIX = [
    {"image_type": "thumbnail", "size": (320, 240)},
    {"image_type": "1080p", "size": (1920, 1080)},
    {"image_type": "4k", "size": (3840, 2160)},
    {"image_type": "large_example", "size": (4000, 3000)},
]

DETAILS = ["low", "high"]
MODEL = "gpt-4.1-mini"

In [4]:
for item in SIZE_MATRIX:
    width, height = item["size"]
    item["image_path"] = str(
        create_synthetic_image(
            IMAGE_DIR / f"{item['image_type']}_{width}x{height}.png",
            size=(width, height),
            label=f"{item['image_type']} - {width}x{height}",
        )
    )

pd.DataFrame(SIZE_MATRIX)

,image_type,size,image_path
0,thumbnail,"(320, 240)",c:\Users\Dell\Documents\KDU-internship\AI\Visi...
1,1080p,"(1920, 1080)",c:\Users\Dell\Documents\KDU-internship\AI\Visi...
2,4k,"(3840, 2160)",c:\Users\Dell\Documents\KDU-internship\AI\Visi...
3,large_example,"(4000, 3000)",c:\Users\Dell\Documents\KDU-internship\AI\Visi...


In [5]:
MEASUREMENT_PROMPT = """
Inspect this image and return JSON only with these keys:
- width_estimate
- height_estimate
- visible_regions
- notes

Keep the response concise. The goal is to force full image ingestion so usage is recorded.
""".strip()

records = []
for item in SIZE_MATRIX:
    for detail in DETAILS:
        parsed, response = call_vision_json(
            client,
            model=MODEL,
            prompt=MEASUREMENT_PROMPT,
            image_path=item["image_path"],
            detail=detail,
            max_output_tokens=500,
        )
        record = build_response_record(
            model=MODEL,
            detail=detail,
            image_path=item["image_path"],
            prompt=MEASUREMENT_PROMPT,
            parsed=parsed,
            response=response,
        )
        record["image_type"] = item["image_type"]
        record["width"] = item["size"][0]
        record["height"] = item["size"][1]
        record["pixels"] = item["size"][0] * item["size"][1]
        records.append(record)

save_json(OUTPUT_DIR / "phase3_token_measurements.json", records)
len(records)

8

In [6]:
rows = []
for record in records:
    usage = record.get("usage", {})
    rows.append(
        {
            "image_type": record["image_type"],
            "width": record["width"],
            "height": record["height"],
            "megapixels": round(record["pixels"] / 1_000_000, 3),
            "detail": record["detail"],
            "input_tokens": usage.get("input_tokens"),
            "output_tokens": usage.get("output_tokens"),
            "total_tokens": usage.get("total_tokens"),
        }
    )

usage_df = pd.DataFrame(rows).sort_values(["width", "detail"])
usage_df

,image_type,width,height,megapixels,detail,input_tokens,output_tokens,total_tokens
1,thumbnail,320,240,0.077,high,184,107,291
0,thumbnail,320,240,0.077,low,184,91,275
3,1080p,1920,1080,2.074,high,2497,117,2614
2,1080p,1920,1080,2.074,low,2497,124,2621
5,4k,3840,2160,8.294,high,2497,104,2601
4,4k,3840,2160,8.294,low,2497,136,2633
7,large_example,4000,3000,12.000,high,2407,133,2540
6,large_example,4000,3000,12.000,low,2407,88,2495


In [7]:
pivot_df = usage_df.pivot(index="image_type", columns="detail", values="input_tokens")
pivot_df

detail,high,low
image_type,,
1080p,2497,2497
4k,2497,2497
large_example,2407,2407
thumbnail,184,184


## Reflection Questions

- `4000x3000` result from this run: the large example used `2407` input tokens, while both `1080p` and `4K` used `2497` input tokens. So token cost did not keep rising linearly with raw pixel count in these measurements.
- Resizing before tiling: this is an inference from the measured outputs, not a direct internals readout. Because `1920x1080` and `3840x2160` both landed at `2497` input tokens, and `4000x3000` landed at `2407`, the model appears to resize or cap large images into a bounded internal representation before tokenization.
- Cost implication: thumbnails were dramatically cheaper at `184` input tokens, while larger images clustered around `2407-2497` input tokens.
- Preprocessing steps that reduce unnecessary token usage: crop whitespace, remove irrelevant borders, isolate the region of interest, and downscale oversized images before upload. These runs also suggest that sparse images with lots of empty space can still consume a large visual budget if left uncropped.